# ORI 2026 — EKSPERIMENTI 1, 2, 3: MLP Predikcija Forex Tržišta
## Eksperiment 1: Predviđanje cena | Eksperiment 2: Predviđanje prinosa | Eksperiment 3: Predviđanje smera trenda
### Svi pod-eksperimenti (1-1/1-2/1-3, 2-1/2-2/2-3, 3-1/3-2/3-3) — MLP Feed-Forward mreža

**Struktura notebooka:**
- 📋 Faza 1: Učitavanje CSV podataka + izveštaj o nedostajućim vrednostima
- 🔧 Faza 2: Inženjering tehničkih indikatora (13 indikatora, paket `ta` + `stockstats`)
- 🧠 Faza 3: Helper funkcije + konfiguracija MLP modela
- 🚀 Faza 4–6: Eksperiment 1 (regresija cena: 1-1, 1-2, 1-3)
- 🚀 Faza 7–9: Eksperiment 2 (regresija prinosa: 2-1, 2-2, 2-3)
- 🚀 Faza 10–12: Eksperiment 3 (klasifikacija smera: 3-1, 3-2 | regresija prinosa: 3-3)
- 📊 Faza 13: Zbirni rezultati + Excel export

## 📋 FAZA 1: Uvoz biblioteka i učitavanje podataka

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
import warnings
import itertools
from math import sqrt
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

warnings.filterwarnings('ignore')

possible_dirs = [".", "data", "./data", "../data"]
INPUT_DIR = ".."
for dir_path in possible_dirs:
    if os.path.exists(dir_path):
        csv_files = glob.glob(os.path.join(dir_path, "*_Data.csv"))
        if csv_files and len(csv_files) >= 5:
            INPUT_DIR = dir_path
            break

print(f"✓ CSV fajlovi pronađeni u: {INPUT_DIR}")
print("✓ Sve biblioteke učitane!")

In [ ]:
def load_csv(path):
    df = pd.read_csv(path, encoding="utf-8-sig")
    df.columns = [c.strip().strip('"') for c in df.columns]
    return df

def clean_numeric_column(series):
    """Konvertuje string kolonu (zarezi/procenti/K,M,B sufiksi) u numeričku."""
    s = series.astype(str).str.replace(",", "", regex=False)
    s = s.str.replace("%", "", regex=False)
    def parse_suffix(val):
        if not isinstance(val, str):
            return np.nan if pd.isna(val) else float(val)
        val = val.strip()
        if val in ("", "nan", "None", "-"):
            return np.nan
        multiplier = 1
        if val.endswith("K"):   multiplier, val = 1_000, val[:-1]
        elif val.endswith("M"): multiplier, val = 1_000_000, val[:-1]
        elif val.endswith("B"): multiplier, val = 1_000_000_000, val[:-1]
        try:    return float(val) * multiplier
        except: return np.nan
    return s.apply(parse_suffix)

csv_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*_Data.csv")))
print(f"Pronađeno {len(csv_files)} CSV fajlova:\n")

all_data = []
missing_report = []

for path in csv_files:
    pair_name = os.path.basename(path).replace("_Data.csv", "")
    df_raw = load_csv(path)
    df_raw["Date"] = pd.to_datetime(df_raw["Date"], format="%m/%d/%Y", errors="coerce")
    numeric_cols = [c for c in ["Price", "Open", "High", "Low", "Change %"] if c in df_raw.columns]
    for col in numeric_cols:
        df_raw[col] = clean_numeric_column(df_raw[col])
    total_rows = len(df_raw)
    cols_to_check = ["Date"] + numeric_cols
    missing_counts = df_raw[cols_to_check].isna().sum()
    missing_pct = (missing_counts / total_rows * 100).round(2)
    report_row = {"Pair": pair_name, "Total_Rows": total_rows}
    for col in cols_to_check:
        report_row[f"{col}_missing_%"] = missing_pct[col]
    missing_report.append(report_row)
    keep_cols = ["Date"] + numeric_cols
    df = df_raw[keep_cols].dropna(subset=["Date", "Price"])
    df["Pair"] = pair_name
    all_data.append(df)
    print(f"  ✓ {pair_name:10s}: {len(df)} redova (od {total_rows} ukupno)")

# Ukloni USDCNY datume kojih nema u ostalim parovima
other_dates = set()
for df in all_data:
    if df["Pair"].iloc[0] != "USDCNY":
        other_dates.update(df["Date"])
for i, df in enumerate(all_data):
    if df["Pair"].iloc[0] == "USDCNY":
        before = len(df)
        all_data[i] = df[df["Date"].isin(other_dates)]
        removed = before - len(all_data[i])
        if removed > 0:
            print(f"\n  ⚠ USDCNY: uklonjeno {removed} redova ({before} → {len(all_data[i])})")

currency_all = pd.concat(all_data, ignore_index=True)
currency_all = currency_all.sort_values(["Pair", "Date"]).reset_index(drop=True)
print(f"\n✓ Ukupno: {len(currency_all)} redova za {currency_all['Pair'].nunique()} parova")

### 📊 Izveštaj o nedostajućim vrednostima po valutnom paru

In [ ]:
missing_df = pd.DataFrame(missing_report)
print("="*80)
print("UDEO NEDOSTAJUĆIH VREDNOSTI PO PARU (%)")
print("="*80 + "\n")
print(missing_df.to_string(index=False))

## 🔧 FAZA 2: Inženjering tehničkih indikatora (paket `ta` + `stockstats`)

In [ ]:
from ta.trend import SMAIndicator, EMAIndicator, MACD, CCIIndicator, PSARIndicator
from ta.momentum import ROCIndicator, RSIIndicator
from ta.volatility import AverageTrueRange
from stockstats import StockDataFrame

INDICATOR_COLS = [
    "SMA_14", "EMA_14", "ROC_14", "RSI_14",
    "MACD", "MACD_Signal", "MACD_Hist",
    "STD_14", "ATR_14",
    "KAMA_14", "SAR", "CCI_14", "CMO_14",
]

def calculate_indicators(pair_df):
    close = pair_df["Price"]
    high  = pair_df["High"] if "High" in pair_df.columns else close
    low   = pair_df["Low"]  if "Low"  in pair_df.columns else close
    out = pd.DataFrame(index=pair_df.index)
    out["SMA_14"]      = SMAIndicator(close=close, window=14).sma_indicator()
    out["EMA_14"]      = EMAIndicator(close=close, window=14).ema_indicator()
    out["ROC_14"]      = ROCIndicator(close=close, window=14).roc()
    out["RSI_14"]      = RSIIndicator(close=close, window=14).rsi()
    macd               = MACD(close=close, window_slow=26, window_fast=12, window_sign=9)
    out["MACD"]        = macd.macd()
    out["MACD_Signal"] = macd.macd_signal()
    out["MACD_Hist"]   = macd.macd_diff()
    out["STD_14"]      = close.rolling(14).std()
    out["ATR_14"]      = AverageTrueRange(high=high, low=low, close=close, window=14).average_true_range()
    out["CCI_14"]      = CCIIndicator(high=high, low=low, close=close, window=14).cci()
    psar               = PSARIndicator(high=high, low=low, close=close, step=0.025, max_step=0.25)
    out["SAR"]         = psar.psar()
    stock = StockDataFrame.retype(pd.DataFrame({
        "close": close.values, "high": high.values, "low": low.values,
        "open": close.values,  "volume": np.ones(len(close)),
    }, index=pair_df.index))
    out["KAMA_14"] = stock["kama_14"]
    out["CMO_14"]  = stock["cmo_14"]
    return out

print("Računanje indikatora...\n")
all_features = []
for pair_name in currency_all["Pair"].unique():
    pair_data  = currency_all[currency_all["Pair"] == pair_name].sort_values("Date").reset_index(drop=True)
    indicators = calculate_indicators(pair_data)
    indicators["Date"]  = pair_data["Date"].values
    indicators["Price"] = pair_data["Price"].values
    indicators["Pair"]  = pair_name
    all_features.append(indicators)
    print(f"  ✓ {pair_name}: {len(indicators)} redova")

features = pd.concat(all_features, ignore_index=True)
features  = features[["Pair", "Date", "Price"] + INDICATOR_COLS]
print(f"\n✓ Shape: {features.shape}")

## 🧠 FAZA 3: Helper funkcije i konfiguracija MLP modela

In [ ]:
from sklearn.neural_network import MLPRegressor, MLPClassifier
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    confusion_matrix, accuracy_score, precision_score, recall_score, f1_score,
)

# ── Globalni parametri ────────────────────────────────────────────────────────
N_LAGS            = 5    # lagovi za cene
N_LAGS_INDICATORS = 2    # lagovi za indikatore (smoothed signali)
TEST_SIZE         = 0.2
ROC_WINDOW        = 14
SMA_WINDOW        = 14

MLP_PARAMS = {
    'hidden_layer_sizes': (64, 32, 16),
    'activation':         'relu',
    'solver':             'adam',
    'learning_rate_init': 0.001,
    'max_iter':           2000,
    'early_stopping':     True,
    'validation_fraction': 0.1,
    'n_iter_no_change':   25,
    'alpha':              0.001,
    'random_state':       42,
    'verbose':            0,
}

# ── Feature transformacije ────────────────────────────────────────────────────
def add_roc(df, cols, n):
    """ROC(n) za svaku kolonu: (x[t]-x[t-n])/x[t-n]*100. Inf → NaN."""
    out = pd.DataFrame(index=df.index)
    for c in cols:
        roc = df[c].pct_change(n) * 100
        out[f"ROC{n}_{c}"] = roc.replace([np.inf, -np.inf], np.nan)
    return out

def add_sma(df, cols, n):
    """SMA(n) za svaku kolonu."""
    out = pd.DataFrame(index=df.index)
    for c in cols:
        out[f"SMA{n}_{c}"] = df[c].rolling(n).mean()
    return out

# ── Target generatori ─────────────────────────────────────────────────────────
def add_price_targets(df, horizons):
    """Eksperiment 1: buduća cena Price[t+h]."""
    out = pd.DataFrame(index=df.index)
    for h in horizons:
        out[f"Target_t+{h}"] = df["Price"].shift(-h)
    return out

def add_return_targets(df, horizons):
    """Eksperiment 2: buduća stopa prinosa ROC(h) = (Price[t+h]-Price[t])/Price[t]*100."""
    out = pd.DataFrame(index=df.index)
    for h in horizons:
        future = df["Price"].shift(-h)
        roc = (future - df["Price"]) / df["Price"].abs().clip(lower=1e-9) * 100
        out[f"ROC_Target_{h}"] = roc.replace([np.inf, -np.inf], np.nan)
    return out

def add_sign_roc_targets(df, horizons):
    """Eksperiment 3: sign(ROC(h)) ∈ {-1, +1}. Nule → +1."""
    out = pd.DataFrame(index=df.index)
    for h in horizons:
        roc = df["Price"].pct_change(h).shift(-h) * 100
        roc = roc.replace([np.inf, -np.inf], np.nan)
        out[f"SignROC_{h}"] = np.sign(roc).replace(0, 1)
    return out

# ── Metrike ───────────────────────────────────────────────────────────────────
def mape_safe(y_true, y_pred, eps=1e-9):
    mask = np.abs(y_true) > eps
    if mask.sum() == 0: return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def direction_accuracy(y_true, y_pred):
    """Procenat pravilno predviđenih smerova promene (Exp1: diff cene)."""
    delta_true = np.diff(y_true)
    delta_pred = np.diff(y_pred)
    if len(delta_true) == 0: return np.nan
    return (np.sign(delta_true) == np.sign(delta_pred)).mean() * 100

# ── Trening funkcije ──────────────────────────────────────────────────────────
def run_regression(X, y, test_size=TEST_SIZE, dates=None, scaler_type='standard'):
    split_idx = int(len(X) * (1 - test_size))
    X_train, X_test = X[:split_idx], X[split_idx:]
    y_train, y_test = y[:split_idx], y[split_idx:]
    if len(X_train) < 30 or len(X_test) < 10: return None
    Scaler    = RobustScaler if scaler_type == 'robust' else StandardScaler
    scaler    = Scaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s  = scaler.transform(X_test)
    model = MLPRegressor(**MLP_PARAMS)
    model.fit(X_train_s, y_train)
    y_pred    = model.predict(X_test_s)
    converged = model.n_iter_ < MLP_PARAMS['max_iter']
    return {
        "MAE": mean_absolute_error(y_test, y_pred),
        "RMSE": float(np.sqrt(mean_squared_error(y_test, y_pred))),
        "R2": r2_score(y_test, y_pred),
        "MAPE": mape_safe(y_test, y_pred),
        "DirAcc": direction_accuracy(y_test, y_pred),
        "Train_Size": len(X_train), "Test_Size": len(X_test),
        "y_test": y_test, "y_pred": y_pred,
        "test_dates": dates[split_idx:] if dates is not None else None,
        "n_iter": model.n_iter_, "converged": converged, "n_features": X.shape[1],
    }

def run_classification(X, y, test_size=TEST_SIZE, dates=None, scaler_type='standard'):
    split_idx = int(len(X) * (1 - test_size))
    X_train, X_test = X[:split_idx], X[split_idx:]
    y_train, y_test = y[:split_idx], y[split_idx:]
    if len(X_train) < 30 or len(X_test) < 10: return None
    if len(np.unique(y_train)) < 2: return None
    Scaler    = RobustScaler if scaler_type == 'robust' else StandardScaler
    scaler    = Scaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s  = scaler.transform(X_test)
    model = MLPClassifier(**MLP_PARAMS)
    model.fit(X_train_s, y_train)
    y_pred    = model.predict(X_test_s)
    converged = model.n_iter_ < MLP_PARAMS['max_iter']
    return {
        "Accuracy":   accuracy_score(y_test, y_pred) * 100,
        "Precision":  precision_score(y_test, y_pred, average='weighted', zero_division=0) * 100,
        "Recall":     recall_score(y_test, y_pred,    average='weighted', zero_division=0) * 100,
        "F1":         f1_score(y_test, y_pred,         average='weighted', zero_division=0) * 100,
        "Train_Size": len(X_train), "Test_Size": len(X_test),
        "y_test": y_test, "y_pred": y_pred,
        "test_dates": dates[split_idx:] if dates is not None else None,
        "n_iter": model.n_iter_, "converged": converged, "n_features": X.shape[1],
    }

# ── Vizuelizacija helper ──────────────────────────────────────────────────────
def plot_confusion_matrix(cm, ax, title, classes=["DOWN", "UP"]):
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)
    ax.imshow(cm_norm, interpolation='nearest', cmap='Blues', vmin=0, vmax=1)
    ax.set_title(title, fontsize=8, pad=4)
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(classes, fontsize=8); ax.set_yticklabels(classes, fontsize=8)
    ax.set_ylabel('Stvarno', fontsize=7); ax.set_xlabel('Predviđeno', fontsize=7)
    for i, j in itertools.product(range(2), range(2)):
        ax.text(j, i, f"{cm[i,j]}\n({cm_norm[i,j]:.0%})",
                ha='center', va='center', fontsize=8,
                color='white' if cm_norm[i, j] > 0.5 else 'black')

def plot_predictions_grid(predictions_dict, subcases, horizons, exp_label, y_label='Cena', h_fmt='t+{}'):
    for pair_name in features['Pair'].unique():
        fig, axes = plt.subplots(len(subcases), len(horizons),
                                  figsize=(5*len(horizons), 3.5*len(subcases)), squeeze=False)
        fig.suptitle(f"Eksperiment {exp_label} — {y_label} — {pair_name}", fontsize=14, fontweight='bold')
        for row, subcase in enumerate(subcases):
            for col, h in enumerate(horizons):
                ax  = axes[row][col]
                key = (pair_name, subcase, h)
                if key not in predictions_dict:
                    ax.set_title(f"{subcase} | {h_fmt.format(h)} (nema podataka)")
                    ax.axis('off'); continue
                data = predictions_dict[key]
                ax.plot(data['dates'], data['y_test'], label='Stvarno',   color='#1f77b4', linewidth=1.5)
                ax.plot(data['dates'], data['y_pred'], label='Predviđeno',color='#d62728', linewidth=1.2, linestyle='--')
                if y_label != 'Cena':
                    ax.axhline(0, color='gray', linewidth=0.7, linestyle=':')
                ax.set_title(f"{subcase} | {h_fmt.format(h)}", fontsize=10)
                ax.xaxis.set_major_locator(mdates.AutoDateLocator())
                ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
                ax.tick_params(axis='x', rotation=30, labelsize=8)
                ax.tick_params(axis='y', labelsize=8)
                ax.grid(alpha=0.3)
                if row == 0 and col == 0:
                    ax.legend(fontsize=8, loc='best')
        plt.tight_layout(rect=[0, 0, 1, 0.96]); plt.show()

def plot_cm_grid(predictions_dict, target_exp, target_sub, exp_label):
    pairs    = sorted({k[0] for k in predictions_dict if k[1] == target_sub})
    horizons = sorted({k[2] for k in predictions_dict if k[1] == target_sub})
    if not pairs or not horizons: return
    fig, axes = plt.subplots(len(pairs), len(horizons),
                              figsize=(3.5*len(horizons), 3.5*len(pairs)), squeeze=False)
    fig.suptitle(f"Eksperiment {exp_label} — Confusion Matrix — {target_sub}", fontsize=13, fontweight='bold')
    for ri, pair in enumerate(pairs):
        for ci, h in enumerate(horizons):
            ax  = axes[ri][ci]
            key = (pair, target_sub, h)
            if key not in predictions_dict: ax.set_visible(False); continue
            yt  = predictions_dict[key]['y_test']
            yp  = predictions_dict[key]['y_pred']
            yt_b = (np.array(yt) > 0).astype(int)
            yp_b = (np.array(yp) > 0).astype(int)
            cm   = confusion_matrix(yt_b, yp_b, labels=[0, 1])
            acc  = (cm[0,0]+cm[1,1]) / cm.sum() * 100
            plot_confusion_matrix(cm, ax, f"{pair} | {h}\nAcc={acc:.1f}%")
    plt.tight_layout(rect=[0, 0, 1, 0.95]); plt.show()

# Zajednička lista rezultata
results_all = []

print(f"✓ MLP parametri: {MLP_PARAMS}")
print(f"✓ N_LAGS={N_LAGS}  N_LAGS_IND={N_LAGS_INDICATORS}  TEST_SIZE={TEST_SIZE}")

## 🚀 FAZA 4–6: EKSPERIMENT 1 — Predviđanje CENA
**Target:** buduća cena `Price[t+h]` — problem regresije

| Eksperiment | Feature ulaz | Horizonti |
|-------------|-------------|----------|
| 1-1 | cene / indikatori / cene+ind | t+1, t+5, t+10 |
| 1-2 | + ROC od ulaznih | t+1, t+3, t+5, t+10 |
| 1-3 | + ROC + SMA od ulaznih | t+1, t+3, t+5, t+10 |

In [ ]:
## 🚀 EKSPERIMENT 1-1

HORIZONS_1_1 = [1, 5, 10]
predictions_1_1 = {}

for pair_name in features['Pair'].unique():
    pdf = features[features['Pair'] == pair_name].sort_values('Date').reset_index(drop=True)

    price_lags = pd.DataFrame(index=pdf.index)
    for lag in range(1, N_LAGS + 1):
        price_lags[f'Price_lag{lag}'] = pdf['Price'].shift(lag)

    indicator_lags = pd.DataFrame(index=pdf.index)
    for col in INDICATOR_COLS:
        indicator_lags[col] = pdf[col]
        for lag in range(1, N_LAGS_INDICATORS + 1):
            indicator_lags[f'{col}_lag{lag}'] = pdf[col].shift(lag)

    targets = add_price_targets(pdf, HORIZONS_1_1)
    feature_sets = {{
        'a) cene':            (price_lags,                                      'standard'),
        'b) indikatori':      (indicator_lags,                                  'robust'),
        'c) cene+indikatori': (pd.concat([price_lags, indicator_lags], axis=1), 'robust'),
    }}

    for subcase, (X_df, scaler_type) in feature_sets.items():
        for h in HORIZONS_1_1:
            combo = pd.concat([X_df, targets[[f'Target_t+{h}']], pdf[['Date']]], axis=1).dropna()
            if len(combo) < 50: continue
            X, y, dates_arr = combo[X_df.columns].values, combo[f'Target_t+{h}'].values, combo['Date'].values
            metrics = run_regression(X, y, dates=dates_arr, scaler_type=scaler_type)
            if metrics is None: continue
            conv_flag = "✓konv." if metrics['converged'] else "⚠NIJE konvergirao"
            print(f"[1-1 {subcase} | {pair_name} | t+{h}] MAE={metrics['MAE']:.5f} RMSE={metrics['RMSE']:.5f} R2={metrics['R2']:.4f} | feat={metrics['n_features']} iter={metrics['n_iter']} {conv_flag}")
            results_all.append({
                'Experiment': '1-1', 'Subcase': subcase, 'Pair': pair_name, 'Horizon': f't+{h}',
                'MAE': metrics['MAE'], 'RMSE': metrics['RMSE'], 'R2': metrics['R2'],
                'MAPE': metrics['MAPE'], 'DirAcc': metrics['DirAcc'],
                'Train_Size': metrics['Train_Size'], 'Test_Size': metrics['Test_Size'],
                'N_Features': metrics['n_features'], 'N_Iter': metrics['n_iter'], 'Converged': metrics['converged'],
            })
            predictions_1_1[(pair_name, subcase, h)] = {
                'dates': metrics['test_dates'], 'y_test': metrics['y_test'], 'y_pred': metrics['y_pred'],
            }

print("\n✓ Eksperiment 1-1 završen.")

In [ ]:
plot_predictions_grid(predictions_1_1,
    ['a) cene', 'b) indikatori', 'c) cene+indikatori'],
    [1, 5, 10], '1-1', 'Cena', 't+{}')

In [ ]:
## 🚀 EKSPERIMENT 1-2

HORIZONS_1_2 = [1, 3, 5, 10]
predictions_1_2 = {}

for pair_name in features['Pair'].unique():
    pdf = features[features['Pair'] == pair_name].sort_values('Date').reset_index(drop=True)

    price_lags = pd.DataFrame(index=pdf.index)
    for lag in range(1, N_LAGS + 1):
        price_lags[f'Price_lag{lag}'] = pdf['Price'].shift(lag)

    indicator_lags = pd.DataFrame(index=pdf.index)
    for col in INDICATOR_COLS:
        indicator_lags[col] = pdf[col]
        for lag in range(1, N_LAGS_INDICATORS + 1):
            indicator_lags[f'{col}_lag{lag}'] = pdf[col].shift(lag)

    roc_price      = add_roc(pdf, ['Price'], ROC_WINDOW)
    roc_indicators = add_roc(pdf, INDICATOR_COLS, ROC_WINDOW)

    targets = add_price_targets(pdf, HORIZONS_1_2)
    feature_sets = {{
        'a) cene+ROC(cena)':    (pd.concat([price_lags, roc_price], axis=1),                                   'standard'),
        'b) ind+ROC(ind)':      (pd.concat([indicator_lags, roc_indicators], axis=1),                          'robust'),
        'c) cene+ind+ROC(oba)': (pd.concat([price_lags, roc_price, indicator_lags, roc_indicators], axis=1),   'robust'),
    }}

    for subcase, (X_df, scaler_type) in feature_sets.items():
        for h in HORIZONS_1_2:
            combo = pd.concat([X_df, targets[[f'Target_t+{h}']], pdf[['Date']]], axis=1).dropna()
            if len(combo) < 50: continue
            X, y, dates_arr = combo[X_df.columns].values, combo[f'Target_t+{h}'].values, combo['Date'].values
            metrics = run_regression(X, y, dates=dates_arr, scaler_type=scaler_type)
            if metrics is None: continue
            conv_flag = "✓konv." if metrics['converged'] else "⚠NIJE konvergirao"
            print(f"[1-2 {subcase} | {pair_name} | t+{h}] MAE={metrics['MAE']:.5f} RMSE={metrics['RMSE']:.5f} R2={metrics['R2']:.4f} | feat={metrics['n_features']} iter={metrics['n_iter']} {conv_flag}")
            results_all.append({
                'Experiment': '1-2', 'Subcase': subcase, 'Pair': pair_name, 'Horizon': f't+{h}',
                'MAE': metrics['MAE'], 'RMSE': metrics['RMSE'], 'R2': metrics['R2'],
                'MAPE': metrics['MAPE'], 'DirAcc': metrics['DirAcc'],
                'Train_Size': metrics['Train_Size'], 'Test_Size': metrics['Test_Size'],
                'N_Features': metrics['n_features'], 'N_Iter': metrics['n_iter'], 'Converged': metrics['converged'],
            })
            predictions_1_2[(pair_name, subcase, h)] = {
                'dates': metrics['test_dates'], 'y_test': metrics['y_test'], 'y_pred': metrics['y_pred'],
            }

print("\n✓ Eksperiment 1-2 završen.")

In [ ]:
plot_predictions_grid(predictions_1_2,
    ['a) cene+ROC(cena)', 'b) ind+ROC(ind)', 'c) cene+ind+ROC(oba)'],
    [1, 3, 5, 10], '1-2', 'Cena', 't+{}')

In [ ]:
## 🚀 EKSPERIMENT 1-3

HORIZONS_1_3 = [1, 3, 5, 10]
predictions_1_3 = {}

for pair_name in features['Pair'].unique():
    pdf = features[features['Pair'] == pair_name].sort_values('Date').reset_index(drop=True)

    price_lags = pd.DataFrame(index=pdf.index)
    for lag in range(1, N_LAGS + 1):
        price_lags[f'Price_lag{lag}'] = pdf['Price'].shift(lag)

    indicator_lags = pd.DataFrame(index=pdf.index)
    for col in INDICATOR_COLS:
        indicator_lags[col] = pdf[col]
        for lag in range(1, N_LAGS_INDICATORS + 1):
            indicator_lags[f'{col}_lag{lag}'] = pdf[col].shift(lag)

    roc_price      = add_roc(pdf, ['Price'], ROC_WINDOW)
    sma_price      = add_sma(pdf, ['Price'], SMA_WINDOW)
    roc_indicators = add_roc(pdf, INDICATOR_COLS, ROC_WINDOW)
    sma_indicators = add_sma(pdf, INDICATOR_COLS, SMA_WINDOW)

    targets = add_price_targets(pdf, HORIZONS_1_3)
    feature_sets = {{
        'a) cene+ROC+SMA(cena)': (pd.concat([price_lags, roc_price, sma_price], axis=1),                                                    'standard'),
        'b) ind+ROC+SMA(ind)':   (pd.concat([indicator_lags, roc_indicators, sma_indicators], axis=1),                                       'robust'),
        'c) sve kombinovano':     (pd.concat([price_lags, roc_price, sma_price, indicator_lags, roc_indicators, sma_indicators], axis=1),     'robust'),
    }}

    for subcase, (X_df, scaler_type) in feature_sets.items():
        for h in HORIZONS_1_3:
            combo = pd.concat([X_df, targets[[f'Target_t+{h}']], pdf[['Date']]], axis=1).dropna()
            if len(combo) < 50: continue
            X, y, dates_arr = combo[X_df.columns].values, combo[f'Target_t+{h}'].values, combo['Date'].values
            metrics = run_regression(X, y, dates=dates_arr, scaler_type=scaler_type)
            if metrics is None: continue
            conv_flag = "✓konv." if metrics['converged'] else "⚠NIJE konvergirao"
            print(f"[1-3 {subcase} | {pair_name} | t+{h}] MAE={metrics['MAE']:.5f} RMSE={metrics['RMSE']:.5f} R2={metrics['R2']:.4f} | feat={metrics['n_features']} iter={metrics['n_iter']} {conv_flag}")
            results_all.append({
                'Experiment': '1-3', 'Subcase': subcase, 'Pair': pair_name, 'Horizon': f't+{h}',
                'MAE': metrics['MAE'], 'RMSE': metrics['RMSE'], 'R2': metrics['R2'],
                'MAPE': metrics['MAPE'], 'DirAcc': metrics['DirAcc'],
                'Train_Size': metrics['Train_Size'], 'Test_Size': metrics['Test_Size'],
                'N_Features': metrics['n_features'], 'N_Iter': metrics['n_iter'], 'Converged': metrics['converged'],
            })
            predictions_1_3[(pair_name, subcase, h)] = {
                'dates': metrics['test_dates'], 'y_test': metrics['y_test'], 'y_pred': metrics['y_pred'],
            }

print("\n✓ Eksperiment 1-3 završen.")

In [ ]:
plot_predictions_grid(predictions_1_3,
    ['a) cene+ROC+SMA(cena)', 'b) ind+ROC+SMA(ind)', 'c) sve kombinovano'],
    [1, 3, 5, 10], '1-3', 'Cena', 't+{}')

## 🚀 FAZA 7–9: EKSPERIMENT 2 — Predviđanje PRINOSA
**Target:** buduća stopa prinosa `ROC(h) = (Price[t+h] - Price[t]) / Price[t] × 100` — problem regresije

| Eksperiment | Feature ulaz | Horizonti |
|-------------|-------------|----------|
| 2-1 | cene / indikatori / cene+ind | ROC(1,3,5,10) |
| 2-2 | + ROC od ulaznih | ROC(1,3,5,10) |
| 2-3 | + ROC + SMA od ulaznih | ROC(1,3,5,10) |

In [ ]:
## 🚀 EKSPERIMENT 2-1

RETURN_HORIZONS = [1, 3, 5, 10]
predictions_2_1 = {}

for pair_name in features['Pair'].unique():
    pdf = features[features['Pair'] == pair_name].sort_values('Date').reset_index(drop=True)

    price_lags = pd.DataFrame(index=pdf.index)
    for lag in range(1, N_LAGS + 1):
        price_lags[f'Price_lag{lag}'] = pdf['Price'].shift(lag)

    indicator_lags = pd.DataFrame(index=pdf.index)
    for col in INDICATOR_COLS:
        indicator_lags[col] = pdf[col]
        for lag in range(1, N_LAGS_INDICATORS + 1):
            indicator_lags[f'{col}_lag{lag}'] = pdf[col].shift(lag)

    targets = add_return_targets(pdf, RETURN_HORIZONS)
    feature_sets = {{
        'a) cene':            (price_lags,                                      'standard'),
        'b) indikatori':      (indicator_lags,                                  'robust'),
        'c) cene+indikatori': (pd.concat([price_lags, indicator_lags], axis=1), 'robust'),
    }}

    for subcase, (X_df, scaler_type) in feature_sets.items():
        for h in RETURN_HORIZONS:
            target_col = f'ROC_Target_{h}'
            combo = pd.concat([X_df, targets[[target_col]], pdf[['Date']]], axis=1).dropna()
            if len(combo) < 50: continue
            X, y, dates_arr = combo[X_df.columns].values, combo[target_col].values, combo['Date'].values
            metrics = run_regression(X, y, dates=dates_arr, scaler_type=scaler_type)
            if metrics is None: continue
            conv_flag = "✓konv." if metrics['converged'] else "⚠NIJE konvergirao"
            print(f"[2-1 {subcase} | {pair_name} | ROC({h})] MAE={metrics['MAE']:.4f} RMSE={metrics['RMSE']:.4f} R2={metrics['R2']:.4f} DirAcc={metrics['DirAcc']:.1f}% | feat={metrics['n_features']} iter={metrics['n_iter']} {conv_flag}")
            results_all.append({
                'Experiment': '2-1', 'Subcase': subcase, 'Pair': pair_name, 'Horizon': f'ROC({h})',
                'MAE': metrics['MAE'], 'RMSE': metrics['RMSE'], 'R2': metrics['R2'],
                'MAPE': metrics['MAPE'], 'DirAcc': metrics['DirAcc'],
                'Train_Size': metrics['Train_Size'], 'Test_Size': metrics['Test_Size'],
                'N_Features': metrics['n_features'], 'N_Iter': metrics['n_iter'], 'Converged': metrics['converged'],
            })
            predictions_2_1[(pair_name, subcase, h)] = {
                'dates': metrics['test_dates'], 'y_test': metrics['y_test'], 'y_pred': metrics['y_pred'],
            }

print("\n✓ Eksperiment 2-1 završen.")

In [ ]:
plot_predictions_grid(predictions_2_1,
    ['a) cene', 'b) indikatori', 'c) cene+indikatori'],
    [1, 3, 5, 10], '2-1', 'Prinos ROC (%)', 'ROC({})')

In [ ]:
## 🚀 EKSPERIMENT 2-2

RETURN_HORIZONS = [1, 3, 5, 10]
predictions_2_2 = {}

for pair_name in features['Pair'].unique():
    pdf = features[features['Pair'] == pair_name].sort_values('Date').reset_index(drop=True)

    price_lags = pd.DataFrame(index=pdf.index)
    for lag in range(1, N_LAGS + 1):
        price_lags[f'Price_lag{lag}'] = pdf['Price'].shift(lag)

    indicator_lags = pd.DataFrame(index=pdf.index)
    for col in INDICATOR_COLS:
        indicator_lags[col] = pdf[col]
        for lag in range(1, N_LAGS_INDICATORS + 1):
            indicator_lags[f'{col}_lag{lag}'] = pdf[col].shift(lag)

    roc_price      = add_roc(pdf, ['Price'], ROC_WINDOW)
    roc_indicators = add_roc(pdf, INDICATOR_COLS, ROC_WINDOW)

    targets = add_return_targets(pdf, RETURN_HORIZONS)
    feature_sets = {{
        'a) cene+ROC(cena)':    (pd.concat([price_lags, roc_price], axis=1),                                   'standard'),
        'b) ind+ROC(ind)':      (pd.concat([indicator_lags, roc_indicators], axis=1),                          'robust'),
        'c) cene+ind+ROC(oba)': (pd.concat([price_lags, roc_price, indicator_lags, roc_indicators], axis=1),   'robust'),
    }}

    for subcase, (X_df, scaler_type) in feature_sets.items():
        for h in RETURN_HORIZONS:
            target_col = f'ROC_Target_{h}'
            combo = pd.concat([X_df, targets[[target_col]], pdf[['Date']]], axis=1).dropna()
            if len(combo) < 50: continue
            X, y, dates_arr = combo[X_df.columns].values, combo[target_col].values, combo['Date'].values
            metrics = run_regression(X, y, dates=dates_arr, scaler_type=scaler_type)
            if metrics is None: continue
            conv_flag = "✓konv." if metrics['converged'] else "⚠NIJE konvergirao"
            print(f"[2-2 {subcase} | {pair_name} | ROC({h})] MAE={metrics['MAE']:.4f} RMSE={metrics['RMSE']:.4f} R2={metrics['R2']:.4f} DirAcc={metrics['DirAcc']:.1f}% | feat={metrics['n_features']} iter={metrics['n_iter']} {conv_flag}")
            results_all.append({
                'Experiment': '2-2', 'Subcase': subcase, 'Pair': pair_name, 'Horizon': f'ROC({h})',
                'MAE': metrics['MAE'], 'RMSE': metrics['RMSE'], 'R2': metrics['R2'],
                'MAPE': metrics['MAPE'], 'DirAcc': metrics['DirAcc'],
                'Train_Size': metrics['Train_Size'], 'Test_Size': metrics['Test_Size'],
                'N_Features': metrics['n_features'], 'N_Iter': metrics['n_iter'], 'Converged': metrics['converged'],
            })
            predictions_2_2[(pair_name, subcase, h)] = {
                'dates': metrics['test_dates'], 'y_test': metrics['y_test'], 'y_pred': metrics['y_pred'],
            }

print("\n✓ Eksperiment 2-2 završen.")

In [ ]:
plot_predictions_grid(predictions_2_2,
    ['a) cene+ROC(cena)', 'b) ind+ROC(ind)', 'c) cene+ind+ROC(oba)'],
    [1, 3, 5, 10], '2-2', 'Prinos ROC (%)', 'ROC({})')

In [ ]:
## 🚀 EKSPERIMENT 2-3

RETURN_HORIZONS = [1, 3, 5, 10]
predictions_2_3 = {}

for pair_name in features['Pair'].unique():
    pdf = features[features['Pair'] == pair_name].sort_values('Date').reset_index(drop=True)

    price_lags = pd.DataFrame(index=pdf.index)
    for lag in range(1, N_LAGS + 1):
        price_lags[f'Price_lag{lag}'] = pdf['Price'].shift(lag)

    indicator_lags = pd.DataFrame(index=pdf.index)
    for col in INDICATOR_COLS:
        indicator_lags[col] = pdf[col]
        for lag in range(1, N_LAGS_INDICATORS + 1):
            indicator_lags[f'{col}_lag{lag}'] = pdf[col].shift(lag)

    roc_price      = add_roc(pdf, ['Price'], ROC_WINDOW)
    sma_price      = add_sma(pdf, ['Price'], SMA_WINDOW)
    roc_indicators = add_roc(pdf, INDICATOR_COLS, ROC_WINDOW)
    sma_indicators = add_sma(pdf, INDICATOR_COLS, SMA_WINDOW)

    targets = add_return_targets(pdf, RETURN_HORIZONS)
    feature_sets = {{
        'a) cene+ROC+SMA(cena)': (pd.concat([price_lags, roc_price, sma_price], axis=1),                                                    'standard'),
        'b) ind+ROC+SMA(ind)':   (pd.concat([indicator_lags, roc_indicators, sma_indicators], axis=1),                                       'robust'),
        'c) sve kombinovano':     (pd.concat([price_lags, roc_price, sma_price, indicator_lags, roc_indicators, sma_indicators], axis=1),     'robust'),
    }}

    for subcase, (X_df, scaler_type) in feature_sets.items():
        for h in RETURN_HORIZONS:
            target_col = f'ROC_Target_{h}'
            combo = pd.concat([X_df, targets[[target_col]], pdf[['Date']]], axis=1).dropna()
            if len(combo) < 50: continue
            X, y, dates_arr = combo[X_df.columns].values, combo[target_col].values, combo['Date'].values
            metrics = run_regression(X, y, dates=dates_arr, scaler_type=scaler_type)
            if metrics is None: continue
            conv_flag = "✓konv." if metrics['converged'] else "⚠NIJE konvergirao"
            print(f"[2-3 {subcase} | {pair_name} | ROC({h})] MAE={metrics['MAE']:.4f} RMSE={metrics['RMSE']:.4f} R2={metrics['R2']:.4f} DirAcc={metrics['DirAcc']:.1f}% | feat={metrics['n_features']} iter={metrics['n_iter']} {conv_flag}")
            results_all.append({
                'Experiment': '2-3', 'Subcase': subcase, 'Pair': pair_name, 'Horizon': f'ROC({h})',
                'MAE': metrics['MAE'], 'RMSE': metrics['RMSE'], 'R2': metrics['R2'],
                'MAPE': metrics['MAPE'], 'DirAcc': metrics['DirAcc'],
                'Train_Size': metrics['Train_Size'], 'Test_Size': metrics['Test_Size'],
                'N_Features': metrics['n_features'], 'N_Iter': metrics['n_iter'], 'Converged': metrics['converged'],
            })
            predictions_2_3[(pair_name, subcase, h)] = {
                'dates': metrics['test_dates'], 'y_test': metrics['y_test'], 'y_pred': metrics['y_pred'],
            }

print("\n✓ Eksperiment 2-3 završen.")

In [ ]:
plot_predictions_grid(predictions_2_3,
    ['a) cene+ROC+SMA(cena)', 'b) ind+ROC+SMA(ind)', 'c) sve kombinovano'],
    [1, 3, 5, 10], '2-3', 'Prinos ROC (%)', 'ROC({})')

## 🚀 FAZA 10–12: EKSPERIMENT 3 — Predviđanje SMERA TRENDA
**Target 3-1/3-2:** `sign(ROC(h))` ∈ {-1, +1} — problem **klasifikacije**

**Target 3-3:** `ROC(h)` — problem **regresije** (prinos kao u Exp2)

| Eksperiment | Feature ulaz | Horizonti | Tip |
|-------------|-------------|----------|-----|
| 3-1 | cene / indikatori / cene+ind | sign(ROC(1,3,5,10)) | Klasifikacija |
| 3-2 | + ROC od ulaznih | sign(ROC(1,3,5,10)) | Klasifikacija |
| 3-3 | + ROC + SMA od ulaznih | ROC(1,3,5,10) | Regresija |

In [ ]:
## 🚀 EKSPERIMENT 3-1

HORIZONS_3 = [1, 3, 5, 10]
predictions_3_1 = {}

for pair_name in features['Pair'].unique():
    pdf = features[features['Pair'] == pair_name].sort_values('Date').reset_index(drop=True)

    price_lags = pd.DataFrame(index=pdf.index)
    for lag in range(1, N_LAGS + 1):
        price_lags[f'Price_lag{lag}'] = pdf['Price'].shift(lag)

    indicator_lags = pd.DataFrame(index=pdf.index)
    for col in INDICATOR_COLS:
        indicator_lags[col] = pdf[col]
        for lag in range(1, N_LAGS_INDICATORS + 1):
            indicator_lags[f'{col}_lag{lag}'] = pdf[col].shift(lag)

    targets = add_sign_roc_targets(pdf, HORIZONS_3)
    feature_sets = {{
        'a) cene':            (price_lags,                                      'standard'),
        'b) indikatori':      (indicator_lags,                                  'robust'),
        'c) cene+indikatori': (pd.concat([price_lags, indicator_lags], axis=1), 'robust'),
    }}

    for subcase, (X_df, scaler_type) in feature_sets.items():
        for h in HORIZONS_3:
            target_col = f'SignROC_{h}'
            combo = pd.concat([X_df, targets[[target_col]], pdf[['Date']]], axis=1).dropna()
            if len(combo) < 50: continue
            X, y, dates_arr = combo[X_df.columns].values, combo[target_col].values, combo['Date'].values
            metrics = run_classification(X, y, dates=dates_arr, scaler_type=scaler_type)
            if metrics is None: continue
            conv_flag = "✓konv." if metrics['converged'] else "⚠NIJE konvergirao"
            print(f"[3-1 {subcase} | {pair_name} | sign(ROC({h}))]  Acc={metrics['Accuracy']:.1f}%  F1={metrics['F1']:.1f}%  feat={metrics['n_features']}  iter={metrics['n_iter']}  {conv_flag}")
            results_all.append({
                'Experiment': '3-1', 'Subcase': subcase, 'Pair': pair_name, 'Horizon': f'sign(ROC({h}))',
                'Accuracy_%':  round(metrics['Accuracy'],  1),
                'Precision_%': round(metrics['Precision'], 1),
                'Recall_%':    round(metrics['Recall'],    1),
                'F1_%':        round(metrics['F1'],        1),
                'Train_Size': metrics['Train_Size'], 'Test_Size': metrics['Test_Size'],
                'N_Features': metrics['n_features'], 'N_Iter': metrics['n_iter'], 'Converged': metrics['converged'],
            })
            predictions_3_1[(pair_name, subcase, h)] = {
                'dates': metrics['test_dates'], 'y_test': metrics['y_test'], 'y_pred': metrics['y_pred'],
            }

print("\n✓ Eksperiment 3-1 završen.")

In [ ]:
plot_cm_grid(predictions_3_1, '3-1', 'c) cene+indikatori', '3-1')

In [ ]:
## 🚀 EKSPERIMENT 3-2

HORIZONS_3 = [1, 3, 5, 10]
predictions_3_2 = {}

for pair_name in features['Pair'].unique():
    pdf = features[features['Pair'] == pair_name].sort_values('Date').reset_index(drop=True)

    price_lags = pd.DataFrame(index=pdf.index)
    for lag in range(1, N_LAGS + 1):
        price_lags[f'Price_lag{lag}'] = pdf['Price'].shift(lag)

    indicator_lags = pd.DataFrame(index=pdf.index)
    for col in INDICATOR_COLS:
        indicator_lags[col] = pdf[col]
        for lag in range(1, N_LAGS_INDICATORS + 1):
            indicator_lags[f'{col}_lag{lag}'] = pdf[col].shift(lag)

    roc_price      = add_roc(pdf, ['Price'], ROC_WINDOW)
    roc_indicators = add_roc(pdf, INDICATOR_COLS, ROC_WINDOW)

    targets = add_sign_roc_targets(pdf, HORIZONS_3)
    feature_sets = {{
        'a) cene+ROC(cena)':    (pd.concat([price_lags, roc_price], axis=1),                                   'standard'),
        'b) ind+ROC(ind)':      (pd.concat([indicator_lags, roc_indicators], axis=1),                          'robust'),
        'c) cene+ind+ROC(oba)': (pd.concat([price_lags, roc_price, indicator_lags, roc_indicators], axis=1),   'robust'),
    }}

    for subcase, (X_df, scaler_type) in feature_sets.items():
        for h in HORIZONS_3:
            target_col = f'SignROC_{h}'
            combo = pd.concat([X_df, targets[[target_col]], pdf[['Date']]], axis=1).dropna()
            if len(combo) < 50: continue
            X, y, dates_arr = combo[X_df.columns].values, combo[target_col].values, combo['Date'].values
            metrics = run_classification(X, y, dates=dates_arr, scaler_type=scaler_type)
            if metrics is None: continue
            conv_flag = "✓konv." if metrics['converged'] else "⚠NIJE konvergirao"
            print(f"[3-2 {subcase} | {pair_name} | sign(ROC({h}))]  Acc={metrics['Accuracy']:.1f}%  F1={metrics['F1']:.1f}%  feat={metrics['n_features']}  iter={metrics['n_iter']}  {conv_flag}")
            results_all.append({
                'Experiment': '3-2', 'Subcase': subcase, 'Pair': pair_name, 'Horizon': f'sign(ROC({h}))',
                'Accuracy_%':  round(metrics['Accuracy'],  1),
                'Precision_%': round(metrics['Precision'], 1),
                'Recall_%':    round(metrics['Recall'],    1),
                'F1_%':        round(metrics['F1'],        1),
                'Train_Size': metrics['Train_Size'], 'Test_Size': metrics['Test_Size'],
                'N_Features': metrics['n_features'], 'N_Iter': metrics['n_iter'], 'Converged': metrics['converged'],
            })
            predictions_3_2[(pair_name, subcase, h)] = {
                'dates': metrics['test_dates'], 'y_test': metrics['y_test'], 'y_pred': metrics['y_pred'],
            }

print("\n✓ Eksperiment 3-2 završen.")

In [ ]:
plot_cm_grid(predictions_3_2, '3-2', 'c) cene+ind+ROC(oba)', '3-2')

In [ ]:
## 🚀 EKSPERIMENT 3-3
### + ROC + SMA od ulaznih → ROC(h) (buduci prinos) — problem regresije

HORIZONS_3_3   = [1, 3, 5, 10]
predictions_3_3 = {}

for pair_name in features['Pair'].unique():
    pdf = features[features['Pair'] == pair_name].sort_values('Date').reset_index(drop=True)

    price_lags = pd.DataFrame(index=pdf.index)
    for lag in range(1, N_LAGS + 1):
        price_lags[f'Price_lag{lag}'] = pdf['Price'].shift(lag)
    roc_price      = add_roc(pdf, ['Price'], ROC_WINDOW)
    sma_price      = add_sma(pdf, ['Price'], SMA_WINDOW)

    indicator_lags = pd.DataFrame(index=pdf.index)
    for col in INDICATOR_COLS:
        indicator_lags[col] = pdf[col]
        for lag in range(1, N_LAGS_INDICATORS + 1):
            indicator_lags[f'{col}_lag{lag}'] = pdf[col].shift(lag)
    roc_indicators = add_roc(pdf, INDICATOR_COLS, ROC_WINDOW)
    sma_indicators = add_sma(pdf, INDICATOR_COLS, SMA_WINDOW)

    targets = add_return_targets(pdf, HORIZONS_3_3)

    feature_sets = {
        'a) cene+ROC+SMA(cena)': (pd.concat([price_lags, roc_price, sma_price], axis=1),                                                  'standard'),
        'b) ind+ROC+SMA(ind)':   (pd.concat([indicator_lags, roc_indicators, sma_indicators], axis=1),                                     'robust'),
        'c) sve kombinovano':     (pd.concat([price_lags, roc_price, sma_price, indicator_lags, roc_indicators, sma_indicators], axis=1),   'robust'),
    }

    for subcase, (X_df, scaler_type) in feature_sets.items():
        for h in HORIZONS_3_3:
            target_col = f'ROC_Target_{h}'
            combo = pd.concat([X_df, targets[[target_col]], pdf[['Date']]], axis=1).dropna()
            if len(combo) < 50: continue
            X, y, dates_arr = combo[X_df.columns].values, combo[target_col].values, combo['Date'].values
            metrics = run_regression(X, y, dates=dates_arr, scaler_type=scaler_type)
            if metrics is None: continue
            conv_flag = "✓konv." if metrics['converged'] else "⚠NIJE konvergirao"
            print(f"[3-3 {subcase} | {pair_name} | ROC({h})]  MAE={metrics['MAE']:.4f}  RMSE={metrics['RMSE']:.4f}  R2={metrics['R2']:.4f}  feat={metrics['n_features']}  iter={metrics['n_iter']}  {conv_flag}")
            results_all.append({
                'Experiment': '3-3', 'Subcase': subcase, 'Pair': pair_name, 'Horizon': f'ROC({h})',
                'MAE': metrics['MAE'], 'RMSE': metrics['RMSE'], 'R2': metrics['R2'],
                'MAPE': metrics['MAPE'], 'DirAcc': metrics['DirAcc'],
                'Train_Size': metrics['Train_Size'], 'Test_Size': metrics['Test_Size'],
                'N_Features': metrics['n_features'], 'N_Iter': metrics['n_iter'], 'Converged': metrics['converged'],
            })
            predictions_3_3[(pair_name, subcase, h)] = {
                'dates': metrics['test_dates'], 'y_test': metrics['y_test'], 'y_pred': metrics['y_pred'],
            }

print("\n✓ Eksperiment 3-3 završen.")

In [ ]:
plot_predictions_grid(predictions_3_3,
    ['a) cene+ROC+SMA(cena)', 'b) ind+ROC+SMA(ind)', 'c) sve kombinovano'],
    [1, 3, 5, 10], '3-3', 'Prinos ROC (%)', 'ROC({})')

## 📊 FAZA 13: Zbirni rezultati svih eksperimenata + Excel export

In [ ]:
results_df_all = pd.DataFrame(results_all)

# ── Regresioni rezultati (Exp 1, 2, 3-3) ──────────────────────────────────────
reg_exps = ['1-1', '1-2', '1-3', '2-1', '2-2', '2-3', '3-3']
reg_cols = ['Experiment', 'Subcase', 'Pair', 'Horizon',
            'MAE', 'RMSE', 'R2', 'MAPE', 'DirAcc',
            'Train_Size', 'Test_Size', 'N_Features', 'N_Iter', 'Converged']

results_reg = results_df_all[results_df_all['Experiment'].isin(reg_exps)][reg_cols]
print("="*110)
print(f"REGRESIONI REZULTATI ({', '.join(reg_exps)}) — {len(results_reg)} kombinacija")
print("="*110 + "\n")
print(results_reg.to_string(index=False))

n_nc = (~results_reg['Converged']).sum()
if n_nc > 0:
    print(f"\n⚠ {n_nc} model(a) NIJE konvergiralo (max_iter={MLP_PARAMS['max_iter']})")

results_reg.to_excel('Experiments_Regression_Results.xlsx', index=False)
print("\n✓ Sačuvano: Experiments_Regression_Results.xlsx")

# ── Klasifikacioni rezultati (3-1, 3-2) ────────────────────────────────────────
clf_exps = ['3-1', '3-2']
clf_cols = ['Experiment', 'Subcase', 'Pair', 'Horizon',
            'Accuracy_%', 'Precision_%', 'Recall_%', 'F1_%',
            'Train_Size', 'Test_Size', 'N_Features', 'N_Iter', 'Converged']

results_clf = results_df_all[results_df_all['Experiment'].isin(clf_exps)][clf_cols]
if not results_clf.empty:
    print("\n" + "="*110)
    print(f"KLASIFIKACIONI REZULTATI ({', '.join(clf_exps)}) — {len(results_clf)} kombinacija")
    print("="*110 + "\n")
    print(results_clf.to_string(index=False))
    n_nc = (~results_clf['Converged']).sum()
    if n_nc > 0:
        print(f"\n⚠ {n_nc} model(a) NIJE konvergiralo.")
    results_clf.to_excel('Experiments_Classification_Results.xlsx', index=False)
    print("\n✓ Sačuvano: Experiments_Classification_Results.xlsx")

In [ ]:
# ── Export pratećih fajlova ────────────────────────────────────────────────────
missing_df.to_excel('00_Missing_Values_Report.xlsx', index=False)
currency_all.to_csv('01_Currency_Data_ALL.csv', index=False)
features.to_csv('02_Features_With_Indicators.csv', index=False)

print("\n✓ Svi fajlovi sačuvani:")
print("  - 00_Missing_Values_Report.xlsx")
print("  - 01_Currency_Data_ALL.csv")
print("  - 02_Features_With_Indicators.csv")
print("  - Experiments_Regression_Results.xlsx   (1-1, 1-2, 1-3, 2-1, 2-2, 2-3, 3-3)")
print("  - Experiments_Classification_Results.xlsx (3-1, 3-2)")
print()
print("="*60)
total = len(results_df_all)
print(f"Ukupno istreniranih modela: {total}")
for exp in ['1-1','1-2','1-3','2-1','2-2','2-3','3-1','3-2','3-3']:
    n = len(results_df_all[results_df_all['Experiment']==exp])
    print(f"  Eksperiment {exp}: {n} modela")
print("="*60)